# 🎓 Eksperimen Bab IV — Sub-bab 4.6 s.d. 4.9
## Preliminary Experiment dengan Data Sintetis (25 Mahasiswa, 4 Sesi)

**Tujuan notebook ini:**
- **4.6** — Preprocessing & feature engineering dari master table sintetis
- **4.7** — Pelatihan Vision Engine (MobileNetV3 proxy) dan Behavioral Engine (GRU proxy)
- **4.8** — Ablation study: B1 (Unimodal Vision) vs B2 (Unimodal Behavioral) vs P (Late Fusion GBM)
- **4.9** — Korelasi engagement prediksi dengan nilai akademik sintetis

**⚠️ Catatan metodologis:**
> Eksperimen ini menggunakan data sintetis sebagai *preliminary validation* pipeline sebelum data kamera riil tersedia.
> Hasil akan diperbarui dengan data riil setelah proses pengumpulan data selesai.

**Langkah:**
1. Upload 3 file CSV ke Colab (atau mount dari Google Drive)
2. Jalankan semua sel dari atas ke bawah
3. Hasil otomatis tersimpan ke `experiment_results.json` dan `narrative_bab4.txt`

## LANGKAH 1: Upload File Data ke Colab

Jalankan sel ini untuk upload 3 file CSV yang sudah dibuat sebelumnya:
- `synthetic_camera_data.csv`
- `synthetic_lms_log.csv`
- `synthetic_master_table.csv`

In [ ]:
# OPSI A: Upload manual
from google.colab import files
print('Upload 3 file CSV: synthetic_camera_data.csv, synthetic_lms_log.csv, synthetic_master_table.csv')
uploaded = files.upload()
print(f'File yang diupload: {list(uploaded.keys())}')

In [ ]:
# OPSI B: Mount dari Google Drive (uncomment jika pakai Drive)
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# shutil.copy('/content/drive/MyDrive/PATH_ANDA/synthetic_camera_data.csv', '.')
# shutil.copy('/content/drive/MyDrive/PATH_ANDA/synthetic_lms_log.csv', '.')
# shutil.copy('/content/drive/MyDrive/PATH_ANDA/synthetic_master_table.csv', '.')
# print('File berhasil disalin dari Drive')

## LANGKAH 2: Instalasi Library

In [ ]:
!pip install imbalanced-learn xgboost lightgbm -q
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np, json, os, time
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.model_selection import GroupKFold
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import (f1_score, accuracy_score, roc_auc_score,
                              precision_score, recall_score, confusion_matrix,
                              ConfusionMatrixDisplay, roc_curve, auc)
from scipy import stats
import xgboost as xgb
import lightgbm as lgb
np.random.seed(42)
print('✅ Semua library berhasil diimport!')
print(f'  sklearn version: {__import__("sklearn").__version__}')
print(f'  XGBoost version: {xgb.__version__}')
print(f'  LightGBM version: {lgb.__version__}')

## LANGKAH 3: Load Data

In [ ]:
df_cam    = pd.read_csv('synthetic_camera_data.csv')
df_log    = pd.read_csv('synthetic_lms_log.csv')
df_master = pd.read_csv('synthetic_master_table.csv')

print('=' * 55)
print('RINGKASAN DATA YANG DIMUAT')
print('=' * 55)
print(f'  Data Kamera   : {len(df_cam):,} baris | {df_cam["user_id"].nunique()} mahasiswa')
print(f'  Log LMS       : {len(df_log):,} baris')
print(f'  Master Table  : {len(df_master):,} baris | {len(df_master.columns)} kolom')
print(f'\n  Distribusi Label Fusion:')
for lbl, cnt in df_master['fusion_label'].value_counts().items():
    print(f'    {lbl:8s}: {cnt:5,} ({cnt/len(df_master)*100:.1f}%)')
print(f'\n  Profil Mahasiswa:')
for prf, cnt in df_master.groupby('profil')['user_id'].nunique().items():
    print(f'    {prf:25s}: {cnt} mahasiswa')

---
## 4.6 PREPROCESSING & FEATURE ENGINEERING

In [ ]:
print('=' * 55)
print('4.6 PREPROCESSING & FEATURE ENGINEERING')
print('=' * 55)

df = df_master.copy()

# ── Statistik missing ──
n_total   = len(df)
n_mc      = int(df['missing_cam'].sum())
n_ml      = int(df['missing_log'].sum())
n_both    = int(((df['missing_cam']==1)&(df['missing_log']==1)).sum())
n_complete= int(((df['missing_cam']==0)&(df['missing_log']==0)).sum())

print(f'\nKetersediaan data per jendela 15 detik:')
print(f'  Total jendela          : {n_total:,}')
print(f'  Lengkap (cam + log)    : {n_complete:,} ({n_complete/n_total*100:.1f}%)')
print(f'  Missing kamera saja    : {n_mc:,} ({n_mc/n_total*100:.1f}%)')
print(f'  Idle window (no log)   : {n_ml:,} ({n_ml/n_total*100:.1f}%)')
print(f'  Missing keduanya       : {n_both:,} ({n_both/n_total*100:.1f}%)')

# ── Imputasi missing kamera dengan median per mahasiswa ──
visual_raw = ['ear_avg','mar','pitch','yaw','roll','gaze_x','gaze_y']
for feat in visual_raw:
    df[feat] = df.groupby('user_id')[feat].transform(
        lambda x: x.fillna(x.median()))
    df[feat] = df[feat].fillna(df[feat].median())

# ── Normalisasi z-score per individu untuk fitur behavioral ──
beh_raw = ['n_actions','action_density','nav_entropy']
for feat in beh_raw:
    df[feat+'_z'] = df.groupby('user_id')[feat].transform(
        lambda x: (x-x.mean())/(x.std()+1e-8))

# ── Moving Average 3 jendela ──
for feat in ['ear_avg','n_actions','action_density']:
    df[feat+'_ma3'] = df.groupby('user_id')[feat].transform(
        lambda x: x.rolling(3,min_periods=1).mean())

# ── Fitur temporal tambahan ──
df['is_morning']   = df['hour'].between(9,12).astype(int)
df['is_afternoon'] = df['hour'].between(13,16).astype(int)
df['is_evening']   = (df['hour']>=20).astype(int)
day_map = {'Monday':1,'Tuesday':2,'Wednesday':3,'Thursday':4,'Friday':5,'Saturday':6,'Sunday':7}
df['day_num'] = df['day_of_week'].map(day_map).fillna(4)

# ── Encode konteks aktivitas ──
ctx_dummies = pd.get_dummies(df['ctx_activity'], prefix='ctx')
df = pd.concat([df, ctx_dummies], axis=1)
ctx_cols = [c for c in df.columns if c.startswith('ctx_') and c!='ctx_activity']

# ── Fitur final ──
VF = ['ear_avg','mar','pitch','yaw','roll','gaze_x','gaze_y',
      'is_drowsy','is_yawning','is_distracted_cam','face_detected',
      'ear_avg_ma3']
BF = ['n_actions','action_density','event_diversity','component_diversity',
      'nav_entropy','idle_flag','n_actions_z','action_density_z',
      'nav_entropy_z','n_actions_ma3','action_density_ma3'] + \
     ctx_cols + ['hour','day_num','is_morning','is_afternoon','is_evening']
AF = VF + BF

for feat in AF:
    if feat in df.columns:
        df[feat] = pd.to_numeric(df[feat], errors='coerce').fillna(0)

print(f'\nFeature Engineering selesai:')
print(f'  Fitur Visual (VF)    : {len(VF)}')
print(f'  Fitur Behavioral (BF): {len(BF)}')
print(f'  Total fitur gabungan : {len(AF)}')

# Simpan preprocessed
df.to_csv('master_preprocessed.csv', index=False, encoding='utf-8-sig')
print(f'\n✅ Master table preprocessed disimpan: {df.shape}')

# ── Visualisasi distribusi missing ──
fig, axes = plt.subplots(1,2,figsize=(12,4))
miss_data = {'Lengkap':n_complete,'Missing Kamera':n_mc,'Idle (no log)':n_ml,'Missing Keduanya':n_both}
axes[0].bar(miss_data.keys(), miss_data.values(), color=['#00A896','#E05A4E','#F5A623','#94A3B8'])
axes[0].set_title('Distribusi Ketersediaan Data per Jendela 15 Detik', fontweight='bold')
axes[0].set_ylabel('Jumlah Jendela')
for i,(k,v) in enumerate(miss_data.items()):
    axes[0].text(i, v+50, f'{v/n_total*100:.1f}%', ha='center', fontsize=10, fontweight='bold')

label_counts = df['fusion_label'].value_counts()
colors_lbl = {'HIGH':'#00A896','MEDIUM':'#F5A623','LOW':'#E05A4E'}
axes[1].pie(label_counts.values, labels=label_counts.index,
            colors=[colors_lbl[l] for l in label_counts.index],
            autopct='%1.1f%%', startangle=90,
            wedgeprops=dict(width=0.55, edgecolor='white', linewidth=2))
axes[1].set_title('Distribusi Label Fusion (Ground Truth)', fontweight='bold')
plt.tight_layout()
plt.savefig('fig_4_6_preprocessing.png', dpi=200, bbox_inches='tight')
plt.show()
print('Gambar 4.6 tersimpan: fig_4_6_preprocessing.png')

---
## 4.7 PELATIHAN SUB-MODEL (Group K-Fold CV, k=5)

In [ ]:
le = LabelEncoder(); le.fit(['LOW','MEDIUM','HIGH'])
df['label_enc'] = le.transform(df['fusion_label'])

Xv = df[VF].values.astype(float)
Xb = df[BF].values.astype(float)
Xa = df[AF].values.astype(float)
y  = df['label_enc'].values
g  = LabelEncoder().fit_transform(df['user_id'])
gkf = GroupKFold(n_splits=5)

LABELS = ['HIGH','LOW','MEDIUM']

def evaluate(name, model, X, y, groups):
    accs,f1s,precs,recs,aucs = [],[],[],[],[]
    cms = np.zeros((3,3),int)
    t0 = time.time()
    for fold_idx,(tr,te) in enumerate(gkf.split(X,y,groups)):
        sc  = StandardScaler()
        Xtr = sc.fit_transform(X[tr])
        Xte = sc.transform(X[te])
        model.fit(Xtr, y[tr])
        yp   = model.predict(Xte)
        ypr  = model.predict_proba(Xte)
        accs.append(accuracy_score(y[te],yp))
        f1s.append(f1_score(y[te],yp,average='macro',zero_division=0))
        precs.append(precision_score(y[te],yp,average='macro',zero_division=0))
        recs.append(recall_score(y[te],yp,average='macro',zero_division=0))
        try:    aucs.append(roc_auc_score(y[te],ypr,multi_class='ovr',average='macro'))
        except: aucs.append(0.5)
        cms += confusion_matrix(y[te],yp,labels=[0,1,2])
        print(f'  Fold {fold_idx+1}/5: F1={f1s[-1]:.4f} AUC={aucs[-1]:.4f}')
    elapsed = time.time()-t0
    result = {
        'name':name,'elapsed':round(elapsed,1),
        'acc':round(float(np.mean(accs)),4),'acc_std':round(float(np.std(accs)),4),
        'f1':round(float(np.mean(f1s)),4),'f1_std':round(float(np.std(f1s)),4),
        'prec':round(float(np.mean(precs)),4),'rec':round(float(np.mean(recs)),4),
        'auc':round(float(np.mean(aucs)),4),'auc_std':round(float(np.std(aucs)),4),
        'cm':cms.tolist()
    }
    print(f'  ✅ {name}: Acc={result["acc"]:.4f}±{result["acc_std"]:.4f} | F1={result["f1"]:.4f}±{result["f1_std"]:.4f} | AUC={result["auc"]:.4f}±{result["auc_std"]:.4f} | [{elapsed:.1f}s]')
    return result

print('='*55)
print('[A] VISION ENGINE — MobileNetV3 proxy (MLP-256-128-64)')
print('    Input: EAR, MAR, Head Pose, Gaze + Moving Average')
print('='*55)
rv_mlp = evaluate('MLP-256-128-64 (proxy MobileNetV3)',
    MLPClassifier(hidden_layer_sizes=(256,128,64),activation='relu',
                  max_iter=500,random_state=42,early_stopping=True,
                  validation_fraction=0.1,n_iter_no_change=15), Xv,y,g)

print('\n'+'-'*40)
print('[Baseline Vision] Random Forest')
rv_rf = evaluate('Random Forest (baseline Vision)',
    RandomForestClassifier(n_estimators=200,random_state=42,n_jobs=-1), Xv,y,g)

best_v = rv_mlp if rv_mlp['f1'] >= rv_rf['f1'] else rv_rf
print(f'\n→ Best Vision Engine: {best_v["name"]} (F1={best_v["f1"]:.4f})')

print('\n'+'='*55)
print('[B] BEHAVIORAL ENGINE — GRU proxy (MLP-128-64-32)')
print('    Input: Log features + Context + Temporal + z-score individual')
print('='*55)
rb_mlp = evaluate('MLP-128-64-32 (proxy GRU)',
    MLPClassifier(hidden_layer_sizes=(128,64,32),activation='relu',
                  max_iter=500,random_state=42,early_stopping=True,
                  validation_fraction=0.1,n_iter_no_change=15), Xb,y,g)

print('\n'+'-'*40)
print('[Baseline Behavioral] XGBoost')
rb_xgb = evaluate('XGBoost (baseline Behavioral)',
    xgb.XGBClassifier(n_estimators=200,max_depth=5,learning_rate=0.1,
                      random_state=42,use_label_encoder=False,
                      eval_metric='mlogloss',verbosity=0), Xb,y,g)

best_b = rb_mlp if rb_mlp['f1'] >= rb_xgb['f1'] else rb_xgb
print(f'\n→ Best Behavioral Engine: {best_b["name"]} (F1={best_b["f1"]:.4f})')

In [ ]:
# ── Visualisasi Confusion Matrix 4.7 ──
fig, axes = plt.subplots(1,2,figsize=(14,5))
for ax, result, title in zip(axes, [best_v, best_b],
    ['Vision Engine\n(proxy MobileNetV3)', 'Behavioral Engine\n(proxy GRU)']):
    cm = np.array(result['cm'])
    cm_norm = cm.astype('float') / (cm.sum(axis=1, keepdims=True)+1e-8)
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=LABELS, yticklabels=LABELS, ax=ax,
                linewidths=0.5, linecolor='white', cbar_kws={'shrink':0.8})
    ax.set_xlabel('Predicted Label', fontsize=11)
    ax.set_ylabel('True Label', fontsize=11)
    ax.set_title(f'{title}\nF1={result["f1"]:.4f} | AUC={result["auc"]:.4f}', fontweight='bold')
plt.suptitle('Gambar 4.5. Confusion Matrix Sub-Model (Group K-Fold CV, k=5, Data Sintetis)',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('fig_4_7_confusion_matrix.png', dpi=200, bbox_inches='tight')
plt.show()

# ── Tabel perbandingan arsitektur ──
print('\n=== Tabel Perbandingan Arsitektur (4.7) ===')
rows = [
    ['Vision MLP-256-128-64',     rv_mlp['acc'], rv_mlp['f1'], rv_mlp['prec'], rv_mlp['rec'], rv_mlp['auc']],
    ['Vision Random Forest',      rv_rf['acc'],  rv_rf['f1'],  rv_rf['prec'],  rv_rf['rec'],  rv_rf['auc']],
    ['Behavioral MLP-128-64-32',  rb_mlp['acc'], rb_mlp['f1'], rb_mlp['prec'], rb_mlp['rec'], rb_mlp['auc']],
    ['Behavioral XGBoost',        rb_xgb['acc'], rb_xgb['f1'], rb_xgb['prec'], rb_xgb['rec'], rb_xgb['auc']],
]
df_tbl = pd.DataFrame(rows, columns=['Model','Accuracy','F1 (macro)','Precision','Recall','AUC-ROC'])
print(df_tbl.to_string(index=False))

---
## 4.8 ABLATION STUDY — Late Fusion Adaptif

In [ ]:
print('='*55)
print('4.8 ABLATION STUDY — 3 SKENARIO KOMPARASI')
print('='*55)

print('\n[B1] Baseline — Unimodal Vision Only')
r_b1 = evaluate('B1 — Unimodal Vision',
    MLPClassifier(hidden_layer_sizes=(256,128,64),activation='relu',
                  max_iter=500,random_state=42,early_stopping=True,
                  validation_fraction=0.1), Xv,y,g)

print('\n[B2] Baseline — Unimodal Behavioral Only')
r_b2 = evaluate('B2 — Unimodal Behavioral',
    MLPClassifier(hidden_layer_sizes=(128,64,32),activation='relu',
                  max_iter=500,random_state=42,early_stopping=True,
                  validation_fraction=0.1), Xb,y,g)

print('\n[P] Proposed — Late Fusion Adaptif (LightGBM Meta-Learner)')
r_p = evaluate('P — Late Fusion Adaptif (LightGBM)',
    lgb.LGBMClassifier(n_estimators=300,learning_rate=0.05,max_depth=6,
                       num_leaves=31,random_state=42,verbosity=-1,
                       n_jobs=-1), Xa,y,g)

d1 = round(r_p['f1']-r_b1['f1'],4)
d2 = round(r_p['f1']-r_b2['f1'],4)

print(f'\n  ΔF1 (P vs B1 Vision)     = {d1:+.4f}')
print(f'  ΔF1 (P vs B2 Behavioral) = {d2:+.4f}')
print(f'  Kriteria ΔF1 ≥ 0.05      : {max(d1,d2) >= 0.05}')

In [ ]:
# ── Visualisasi Ablation Study ──
fig, axes = plt.subplots(1,3,figsize=(16,5))
scenarios = [('B1\nUnimodal Vision',r_b1,'#94A3B8'),
             ('B2\nUnimodal Behavioral','B2',r_b2),
             ('P\nLate Fusion Adaptif',r_p,'#00A896')]
scenarios = [('B1\nUnimodal\nVision',r_b1,'#94A3B8'),
             ('B2\nUnimodal\nBehavioral',r_b2,'#0891B2'),
             ('P\nLate Fusion\nAdaptif',r_p,'#00A896')]

metrics = ['acc','f1','auc']
metric_labels = ['Accuracy','F1-Score\n(macro)','AUC-ROC']

for ax, (sc_name, sc_res, sc_color) in zip(axes, scenarios):
    vals = [sc_res[m] for m in metrics]
    bars = ax.bar(metric_labels, vals, color=sc_color, alpha=0.85,
                  edgecolor='white', linewidth=1.5)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                f'{val:.4f}', ha='center', fontsize=11, fontweight='bold')
    ax.set_ylim(0, 1.1)
    ax.set_title(sc_name, fontweight='bold', fontsize=12)
    ax.axhline(0.70, color='red', linestyle='--', linewidth=1.5, alpha=0.5, label='Threshold 0.70')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Gambar 4.6. Perbandingan Performa Tiga Skenario Ablation Study\n(Group K-Fold CV, k=5, Data Sintetis)',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig_4_8_ablation.png', dpi=200, bbox_inches='tight')
plt.show()

# ── Tabel perbandingan ──
print('\n=== Tabel Perbandingan Ablation Study (4.8) ===')
rows_abl = [
    ['B1 — Unimodal Vision',     r_b1['acc'],r_b1['acc_std'],r_b1['f1'],r_b1['f1_std'],r_b1['prec'],r_b1['rec'],r_b1['auc'],r_b1['auc_std'],'—'],
    ['B2 — Unimodal Behavioral', r_b2['acc'],r_b2['acc_std'],r_b2['f1'],r_b2['f1_std'],r_b2['prec'],r_b2['rec'],r_b2['auc'],r_b2['auc_std'],'—'],
    ['P — Late Fusion Adaptif',  r_p['acc'], r_p['acc_std'], r_p['f1'], r_p['f1_std'], r_p['prec'], r_p['rec'], r_p['auc'], r_p['auc_std'],f'+{max(d1,d2):.4f}'],
]
df_abl = pd.DataFrame(rows_abl, columns=['Skenario','Acc','±','F1','±','P','R','AUC','±','ΔF1'])
print(df_abl.to_string(index=False))

---
## 4.9 KORELASI ENGAGEMENT – NILAI AKADEMIK

In [ ]:
print('='*55)
print('4.9 KORELASI ENGAGEMENT – NILAI AKADEMIK')
print('='*55)

# Nilai sintetis berdasarkan profil
profil_nilai = {
    'highly_focused':    (82, 8),
    'normal':            (72, 10),
    'distracted':        (58, 12),
    'fatigued':          (62, 11),
    'extreme_distracted':(45, 8),
}

mhs = df.groupby(['user_id','profil']).agg(
    es_score=('label_enc','mean'),
    pct_high=('fusion_label', lambda x:(x=='HIGH').mean()),
    pct_low=('fusion_label',  lambda x:(x=='LOW').mean()),
    pct_med=('fusion_label',  lambda x:(x=='MEDIUM').mean()),
).reset_index()

np.random.seed(42)
vals = []
for _, row in mhs.iterrows():
    mu, sd = profil_nilai[row['profil']]
    bonus  = (row['es_score'] - 0.5) * 20
    vals.append(round(float(np.clip(np.random.normal(mu+bonus, sd), 20, 100)), 1))
mhs['nilai_kuis'] = vals

# Korelasi
rc,  pc   = stats.pearsonr(mhs['es_score'],  mhs['nilai_kuis'])
rhoc, psc = stats.spearmanr(mhs['es_score'], mhs['nilai_kuis'])

print(f'\n  n mahasiswa        : {len(mhs)}')
print(f'  Pearson r          = {rc:.4f}  (p = {pc:.4f}, {"Signifikan" if pc<0.05 else "Tidak Signifikan"})')
print(f'  Spearman ρ         = {rhoc:.4f} (p = {psc:.4f})')
print(f'\n  Per profil:')
for pr, grp in mhs.groupby('profil'):
    print(f'    {pr:25s}: Nilai={grp["nilai_kuis"].mean():.1f}  ES={grp["es_score"].mean():.3f}')

# Scatter plot
fig, axes = plt.subplots(1,2,figsize=(14,5))
color_map = {'highly_focused':'#00A896','normal':'#0891B2','distracted':'#F5A623',
             'fatigued':'#E05A4E','extreme_distracted':'#7C3AED'}

ax1 = axes[0]
for profil, grp in mhs.groupby('profil'):
    ax1.scatter(grp['es_score'], grp['nilai_kuis'],
                color=color_map[profil], s=80, alpha=0.85,
                label=profil.replace('_',' ').title(), edgecolor='white', linewidth=0.5)
z = np.polyfit(mhs['es_score'], mhs['nilai_kuis'], 1)
p_line = np.poly1d(z)
x_range = np.linspace(mhs['es_score'].min(), mhs['es_score'].max(), 100)
ax1.plot(x_range, p_line(x_range), 'r--', linewidth=2.5, alpha=0.8, label='Regression line')
ax1.set_xlabel('ES Prediksi (rata-rata per mahasiswa)', fontsize=11)
ax1.set_ylabel('Nilai Kuis Sintetis', fontsize=11)
ax1.set_title(f'Korelasi ES – Nilai Kuis\nr = {rc:.4f} (p = {pc:.4f})', fontweight='bold')
ax1.legend(fontsize=9, loc='upper left')
ax1.grid(True, alpha=0.3)
ax1.text(0.97, 0.05, f'r = {rc:.4f}\np = {pc:.4f}\nρ = {rhoc:.4f}',
         transform=ax1.transAxes, ha='right', va='bottom', fontsize=11,
         bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

ax2 = axes[1]
grp_data = [mhs[mhs['profil']==p]['nilai_kuis'].values for p in profil_nilai.keys()]
bp = ax2.boxplot(grp_data, patch_artist=True,
                 medianprops=dict(color='#1F3864', linewidth=2.5))
colors_bp = list(color_map.values())
for patch, color in zip(bp['boxes'], colors_bp):
    patch.set_facecolor(color); patch.set_alpha(0.8)
ax2.set_xticks(range(1,6))
ax2.set_xticklabels([p.replace('_',' ').replace('highly focused','Sangat\nFokus').replace('normal','Normal').replace('distracted','Distracted').replace('fatigued','Fatigued').replace('extreme distracted','Extreme\nDistracted')
                     for p in profil_nilai.keys()], fontsize=9)
ax2.set_ylabel('Nilai Kuis Sintetis', fontsize=11)
ax2.set_title('Distribusi Nilai per Profil Engagement', fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

plt.suptitle('Gambar 4.7. Korelasi Engagement Score – Nilai Akademik (n=25, Data Sintetis)',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig_4_9_correlation.png', dpi=200, bbox_inches='tight')
plt.show()

---
## LANGKAH AKHIR: Simpan Semua Hasil & Generate Narasi

In [ ]:
# ── Simpan semua hasil ke JSON ──
results = {
    'metadata': {'data_type':'synthetic','n_mahasiswa':25,'n_windows':n_total,'n_folds':5,'seed':42},
    'preprocessing': {'n_total':n_total,'n_complete':n_complete,'pct_miss_cam':round(n_mc/n_total*100,1),'pct_miss_log':round(n_ml/n_total*100,1),'n_feat_visual':len(VF),'n_feat_behavioral':len(BF),'n_feat_total':len(AF)},
    'vision_engine': {k:v for k,v in best_v.items()},
    'behavioral_engine': {k:v for k,v in best_b.items()},
    'all_vision_models': {'mlp':rv_mlp,'rf':rv_rf},
    'all_behavioral_models': {'mlp':rb_mlp,'xgb':rb_xgb},
    'fusion': {'B1':r_b1,'B2':r_b2,'P':r_p,'delta_f1_vs_B1':d1,'delta_f1_vs_B2':d2},
    'correlation': {'pearson_r':round(float(rc),4),'p_value':round(float(pc),4),'spearman_rho':round(float(rhoc),4),'p_spearman':round(float(psc),4),'significant':bool(pc<0.05),'n':int(len(mhs))},
    'mhs_stats': mhs.round(4).to_dict(orient='records')
}
with open('experiment_results.json','w') as f: json.dump(results,f,indent=2)
mhs.to_csv('mhs_correlation_results.csv', index=False, encoding='utf-8-sig')
print('✅ experiment_results.json tersimpan')
print('✅ mhs_correlation_results.csv tersimpan')

# ── Generate narasi otomatis ──
narasi = f"""
=================================================================
NARASI OTOMATIS — BAB IV SUB-BAB 4.6 HINGGA 4.9
Berdasarkan Preliminary Experiment dengan Data Sintetis
(25 mahasiswa, 4 sesi, 13.908 jendela 15 detik)
=================================================================

4.6 PREPROCESSING & FEATURE ENGINEERING
Total jendela 15 detik: {n_total:,}
Jendela lengkap (kamera + log): {n_complete:,} ({n_complete/n_total*100:.1f}%)
Jendela idle (no log): {n_ml:,} ({n_ml/n_total*100:.1f}%)
Missing kamera: {n_mc:,} ({n_mc/n_total*100:.1f}%)
Total fitur: {len(AF)} ({len(VF)} visual + {len(BF)} behavioral)
Strategi imputation: median per mahasiswa (visual), z-score individu (behavioral)
Moving Average: 3 jendela untuk EAR, n_actions, action_density

4.7 PELATIHAN SUB-MODEL
Vision Engine ({best_v['name']}):
  Accuracy  = {best_v['acc']:.4f} ± {best_v['acc_std']:.4f}
  F1-Score  = {best_v['f1']:.4f} ± {best_v['f1_std']:.4f}
  AUC-ROC   = {best_v['auc']:.4f} ± {best_v['auc_std']:.4f}
  Precision = {best_v['prec']:.4f}
  Recall    = {best_v['rec']:.4f}

Behavioral Engine ({best_b['name']}):
  Accuracy  = {best_b['acc']:.4f} ± {best_b['acc_std']:.4f}
  F1-Score  = {best_b['f1']:.4f} ± {best_b['f1_std']:.4f}
  AUC-ROC   = {best_b['auc']:.4f} ± {best_b['auc_std']:.4f}
  Precision = {best_b['prec']:.4f}
  Recall    = {best_b['rec']:.4f}

4.8 ABLATION STUDY
B1 (Unimodal Vision):
  Accuracy={r_b1['acc']:.4f} F1={r_b1['f1']:.4f} AUC={r_b1['auc']:.4f}
B2 (Unimodal Behavioral):
  Accuracy={r_b2['acc']:.4f} F1={r_b2['f1']:.4f} AUC={r_b2['auc']:.4f}
P (Late Fusion Adaptif — LightGBM):
  Accuracy={r_p['acc']:.4f} F1={r_p['f1']:.4f} AUC={r_p['auc']:.4f}
Delta F1 vs B1 = {d1:+.4f}
Delta F1 vs B2 = {d2:+.4f}
Kriteria terpenuhi (dF1>=0.05): {max(d1,d2)>=0.05}

4.9 KORELASI ENGAGEMENT - PRESTASI
n mahasiswa = {len(mhs)}
Pearson r   = {rc:.4f} (p = {pc:.4f}, {'SIGNIFIKAN' if pc<0.05 else 'TIDAK SIGNIFIKAN'})
Spearman rho = {rhoc:.4f} (p = {psc:.4f})

=================================================================
CATATAN: Semua hasil ini berbasis data SINTETIS.
Akan diperbarui dengan data riil setelah data kamera tersedia.
=================================================================
"""

with open('narrative_bab4.txt','w', encoding='utf-8') as f:
    f.write(narasi)
print(narasi)
print('✅ narrative_bab4.txt tersimpan')

In [ ]:
# ── Download semua output ──
from google.colab import files
for fname in ['experiment_results.json','narrative_bab4.txt','mhs_correlation_results.csv',
              'fig_4_6_preprocessing.png','fig_4_7_confusion_matrix.png',
              'fig_4_8_ablation.png','fig_4_9_correlation.png',
              'master_preprocessed.csv']:
    if os.path.exists(fname):
        files.download(fname)
        print(f'✅ Downloaded: {fname}')
    else:
        print(f'⚠️  Not found: {fname}')